<a href="https://colab.research.google.com/github/Guilherme5G/GPT_from_scratch/blob/main/gpt_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters: ", len(text))
print(text[:1000])

--2026-11-05 18:33:01--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-11-05 18:33:01 (22.2 MB/s) - ‘input.txt’ saved [1115394/1115394]

length of dataset in characters:  1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a ver

In [2]:
chars = sorted(list(set(text)))
vocab_size = len(chars)

print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [8]:
#criando um mapping dos caracteres para int
stoi = {ch:i for i,ch in enumerate(chars)} # string to integer
itos = {i:ch for i,ch in enumerate(chars)} # integer to string

# ENCODER:
# lambda s: cria uma função que recebe uma string (s).
# for c in s: percorre cada caractere da string.
# stoi[c]: busca o ID numérico correspondente ao caractere.
# []: constrói uma lista com esses IDs.
# Exemplo: "cab" -> [2, 0, 1] (usando o vocabulário do exemplo).
encode = lambda s: [stoi[c] for c in s] #encoder: pega uma string, output uma lista de ints

# DECODER:
# lambda l: cria uma função que recebe uma lista de IDs (l).
# for i in l: percorre cada ID da lista.
# itos[i]: recupera o caractere correspondente.
# ''.join(): junta os caracteres em uma única string.
# Exemplo: [2, 0, 1] -> "cab".
decode = lambda l: ''.join([itos[i] for i in l]) #decoder: pega a lista de ints, output uma string

print(encode("hii there"))
print(decode(encode("hii there")))

[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [7]:
# Convertemos todo o texto em IDs numéricos usando nosso encoder (stoi).
# O GPT não trabalha diretamente com strings, mas com representações numéricas.
# torch.tensor() transforma esses IDs em um tensor do PyTorch.
# dtype=torch.long define os valores como inteiros de 64 bits (int64).
import torch

data = torch.tensor(encode(text), dtype=torch.long)

# shape mostra as dimensões do tensor.
# dtype mostra o tipo dos dados armazenados.
print(data.shape, data.dtype)

# Exibe os primeiros 1000 tokens (neste caso, caracteres codificados).
# Assim visualizamos como o texto foi representado numericamente.
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

Estamos convertendo todo o dataset de texto em uma sequência de identificadores numéricos e armazenando essa sequência em um tensor PyTorch. Isso prepara os dados para o treinamento do GPT. O print permite visualizar os IDs dos primeiros 1.000 caracteres e verificar o formato e o tipo do tensor.

==================================================================

In [10]:
# Separamos o dataset em treino (90%) e validação (10%).

# len(data) retorna o total de tokens.
# Multiplicamos por 0.9 e usamos int() para obter um índice inteiro.
n = int(0.9 * len(data))

# [:n] pega os primeiros n tokens, sem incluir o índice n.
train_data = data[:n]

# [n:] pega do índice n (inclusive) até o final.
val_data = data[n:]

In [11]:
# Define o tamanho máximo do contexto que o modelo
# utilizará para prever o próximo token.
block_size = 8

# Selecionamos os primeiros 9 tokens do treino.
# São 8 tokens de contexto + 1 token que servirá como
# resposta esperada para a última previsão.
# Cada número é o ID de um caractere, não uma coordenada.
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [12]:
# x contém os primeiros 8 tokens (entradas).
# y contém os mesmos dados deslocados uma posição para frente,
# pois queremos aprender a prever o próximo token.
x = train_data[:block_size]
y = train_data[1:block_size+1]

# Percorremos as 8 posições do bloco.
for t in range(block_size):

    # Seleciona os primeiros t+1 tokens como contexto.
    # O contexto cresce: 1 token, 2 tokens, 3 tokens... até 8.
    context = x[:t+1]

    # Seleciona o próximo token correto para esse contexto.
    # É o valor que queremos que o GPT aprenda a prever.
    target = y[t]

    # Mostra a entrada e sua resposta esperada.
    print(f"when input is {context} the target: {target}")

when input is tensor([18]) the target: 47
when input is tensor([18, 47]) the target: 56
when input is tensor([18, 47, 56]) the target: 57
when input is tensor([18, 47, 56, 57]) the target: 58
when input is tensor([18, 47, 56, 57, 58]) the target: 1
when input is tensor([18, 47, 56, 57, 58,  1]) the target: 15
when input is tensor([18, 47, 56, 57, 58,  1, 15]) the target: 47
when input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target: 58


In [14]:
# Define uma semente para a geração de números aleatórios.
# Isso ajuda a reproduzir os mesmos resultados ao executar o código
# novamente, desde que a sequência de operações seja a mesma.
torch.manual_seed(1337)

# Quantas sequências independentes serão processadas em paralelo.
batch_size = 4

# Quantos tokens cada sequência de entrada terá.
# Também representa o contexto máximo do modelo neste exemplo.
block_size = 8

def get_batch(split):
    # Gera um pequeno lote (batch) de entradas x e targets y.

    # Escolhe qual dataset utilizar:
    # 'train' -> dados de treinamento
    # qualquer outro valor -> dados de validação
    data = train_data if split == 'train' else val_data

    # Sorteia 4 índices para começar a extrair sequências do dataset.
    # len(data) - block_size garante espaço para a entrada
    # e para o próximo token que será usado como target.
    # (batch_size,) define o formato do tensor: (4,).
    ix = torch.randint(len(data) - block_size, (batch_size,))

    # Para cada índice sorteado, pega 8 tokens consecutivos.
    # torch.stack() junta as 4 sequências em um tensor 2D (4, 8).
    x = torch.stack([data[i:i+block_size] for i in ix])

    # Faz o mesmo para os targets, mas começando uma posição à frente.
    # Assim, cada posição de y contém o próximo token de x.
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    # Retorna as entradas e as respostas esperadas.
    return x, y


# Gera um batch usando o conjunto de treinamento.
xb, yb = get_batch('train')

# Mostra as entradas e seu formato.
print('inputs:')
print(xb.shape)
print(xb)

# Mostra os targets e seu formato.
print('targets:')
print(yb.shape)
print(yb)

print('----')

# Percorre cada uma das 4 sequências do batch.
for b in range(batch_size):  # batch dimension

    # Percorre as 8 posições de cada sequência.
    for t in range(block_size):  # time dimension

        # Seleciona a sequência b e os tokens de 0 até t, inclusive.
        context = xb[b, :t+1]

        # Seleciona o próximo token correto daquela posição.
        target = yb[b, t]

        # Mostra cada contexto e seu respectivo target.
        print(f"when input is {context.tolist()} the target: {target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
when input is [24] the target: 43
when input is [24, 43] the target: 58
when input is [24, 43, 58] the target: 5
when input is [24, 43, 58, 5] the target: 57
when input is [24, 43, 58, 5, 57] the target: 1
when input is [24, 43, 58, 5, 57, 1] the target: 46
when input is [24, 43, 58, 5, 57, 1, 46] the target: 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target: 39
when input is [44] the target: 53
when input is [44, 53] the target: 56
when input is [44, 53, 56] the target: 1
when input is [44, 53, 56, 1] the target: 58
when input is [44, 53, 56, 1, 58] the target: 46
when input is [44, 53

In [16]:
print(xb)

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [27]:
import torch
import torch.nn as nn  # Ferramentas do PyTorch para criar redes neurais.
from torch.nn import functional as F  # Funções como softmax e cross-entropy.
torch.manual_seed(1337)  # Semente para reproduzir a inicialização aleatória.

# Criamos nosso primeiro modelo de linguagem: Bigram.
# Ele utiliza apenas o token atual para prever o próximo.
# nn.Module permite criar um modelo com parâmetros treináveis.
class BigramLanguageModel(nn.Module):

    # Construtor: executado quando criamos o modelo.
    # vocab_size = quantidade de caracteres diferentes (65).
    def __init__(self, vocab_size):
        super().__init__()

        # Cria uma tabela treinável de tamanho (65, 65).
        # Para cada token, retorna 65 logits (scores),
        # um para cada possível próximo token.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    # Define o que acontece quando executamos o modelo.
    # idx = tokens de entrada (xb).
    # targets = tokens corretos que queremos prever (yb).
    def forward(self, idx, targets=None):

        # idx e targets possuem formato (B, T).
        # B = número de sequências do batch (4).
        # T = tokens por sequência (8).

        # Para cada ID, consulta a linha correspondente
        # na tabela e retorna os scores dos próximos tokens.
        # Formato: (B, T, C), com C = vocab_size (65).
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None

        else:
            #Obtem as dimensoes dos logits.
            B, T, C = logits.shape

            # Transforma (B, T, C) em (B*T, C).
            # Exemplo: (4, 8, 65) -> (32, 65).
            # São 32 previsões, cada uma com 65 scores.
            logits = logits.view(B*T, C)

            # Transforma (B, T) em (B*T).
            # Exemplo: (4, 8) -> (32,).
            # Cada elemento é o ID do próximo token correto.
            targets = targets.view(B*T)

            # Calcula o erro médio das 32 previsões usando Cross-Entropy.
            loss = F.cross_entropy(logits, targets)

        # Retorna os logits reorganizados e a loss calculada.
        # Ainda não estamos atualizando os pesos do modelo
        return logits, loss


    def generate(self, idx, max_new_tokens):
        # idx possui formato (B, T).
        # B = número de sequências.
        # T = quantidade de tokens no contexto atual.

        # Repete o processo para gerar a quantidade solicitada de tokens.
        for _ in range(max_new_tokens):

            # Passa o contexto atual pelo modelo.
            # Retorna os logits das previsões e a loss (None na geração).
            logits, loss = self(idx)

            # Seleciona somente os logits da última posição da sequência.
            # (B, T, C) -> (B, C)
            # C = quantidade de possíveis próximos tokens.
            logits = logits[:, -1, :]

            # Transforma os logits em probabilidades com softmax.
            # As probabilidades somam 1 para cada sequência.
            probs = F.softmax(logits, dim=-1)

            # Sorteia o próximo token conforme essas probabilidades.
            # Retorna um ID para cada sequência: (B, 1).
            idx_next = torch.multinomial(probs, num_samples=1)

            # Adiciona o novo token ao final de cada sequência.
            # O contexto aumenta de (B, T) para (B, T+1).
            idx = torch.cat((idx, idx_next), dim=1) # concatenar: juntar tensores.

        # Retorna as sequências originais com os novos tokens gerados.
        return idx


# Cria uma instância do modelo usando nosso vocabulário.
m = BigramLanguageModel(vocab_size)

# Executa o modelo com as entradas xb e os targets yb.
# O PyTorch chama o forward() automaticamente.
logits, loss = m(xb, yb)

# Mostra o formato dos logits: (4, 8, 65).
print(logits.shape)
print(loss)
'''idx = torch.zeros((1, 1), dtype=torch.long)
generated = m.generate(idx, max_new_tokens=100)
print(decode(generated[0].lolist()))'''
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ
